# <b style='font-size:30px;font-family:Arial'>File-Embedding-Based Collection — Multiple JSON Files</b>

This notebook demonstrates two ways to ingest multiple files into a Teradata Vector Store collection, and walks through a real-world failure scenario where one file is corrupt (it uses a wrong metadata column name internally) and how to recover from it by adding a corrected replacement file:

| Approach | When to use |
|---|---|
| **Ingestor Pipeline** `.files().upsert().run()` | Creating a new collection from scratch. One call handles everything: collection creation, chunking, embedding, and indexing. |
| **Empty Ingestor + `add_documents`** | You already have an empty collection and want to append files to it. Useful when adding data incrementally to an existing collection — including appending a fixed replacement for a previously failed file. |


---
## <b style='font-size:22px;font-family:Arial'>1 · Import Required Libraries</b>

In [3]:
import os
import json
import glob
import getpass
from datetime import datetime

from teradataml import create_context, DataFrame
from teradatagenai import Collection, CollectionManager, ColumnInfo
from teradatagenai import BasicIngestor, ExtractionSchema
from teradatagenai import LocalConfig, TeradataAI
from teradatagenai.vector_store.Ingestor import Ingestor
from teradatagenai.common.constants import CollectionType
from teradatasqlalchemy.types import VARCHAR

---
## <b style='font-size:22px;font-family:Arial'>2 · Connect to Teradata Vantage</b>

In [ ]:
# Configure connection parameters
hostname = getpass.getpass('Enter Teradata Host: ')
username = getpass.getpass('Enter Teradata Username: ')
password = getpass.getpass('Enter Teradata Password: ')

# Create database context
context=create_context(host=hostname, username=username, password=password)
print("✅ Database connection established")
print(f"🌐 Connected to: {hostname}")
print(f"👤 User: {username}")

In [ ]:
from teradataml import set_auth_token

# Refer to help(set_auth_token) for more details on how to set the auth token for your environment.
# Uncomment the below lines if you want to set authentication token using basic authentication.
# You will be prompted to enter the base_url for your environment.

# base_url = getpass.getpass('Enter Base URL: ')
# set_auth_token(base_url= base_url, 
#                username = username, 
#                password = password, 
#                auth_mech = "BASIC")

In [ ]:
# Verify service health
try:
    health = CollectionManager.health()
except Exception as e:
    print("❌ Collection service health check failed:",e)

---
## <b style='font-size:22px;font-family:Arial'>3 · Configure Embedding Model</b>

In [4]:
embedding_model = TeradataAI(
    api_type="td_hosted",
    model_name="amazon.titan-embed-text-v1",
)

---
## <b style='font-size:22px;font-family:Arial'>4 · Discover Files, Configure LocalConfig & ExtractionSchema</b>

These two objects are shared by both ingestion approaches below.

The JSON records in this dataset have the structure:
```json
{ "text": "...", "embeddings": [...1536 floats...], "element_id": "...", "metadata": { "filename": "...", "page_number": 1, ... } }
```
The `ExtractionSchema` maps `text` as the data column and `embeddings` as the pre-computed vector column — making this a **FILE_EMBEDDING_BASED** collection (no embedding generation needed at runtime).

In [1]:
import os, glob
notebook_dir = os.getcwd()
JSON_DIR = os.path.join(os.path.dirname(notebook_dir), "example-data", "json_files", "multiple_json")

json_files = sorted(glob.glob(os.path.join(JSON_DIR, "*.json")))
print(f"Found {len(json_files)} JSON files")

Found 25 JSON files


In [ ]:
local_config = LocalConfig(
    files=json_files,
    files_type="json"
)

extraction_schema = ExtractionSchema(
    data_columns=[
        ColumnInfo(name="text", datatype=VARCHAR(32000))
    ],
    embedding_columns=[
        ColumnInfo(name="embeddings")
    ],
    metadata_columns = [
        ColumnInfo(name="languages")
    ]
)

---
## <b style='font-size:22px;font-family:Arial'>Approach 1 — Ingestor Pipeline</b>

Build the pipeline with `.files()` → `.upsert()` → `.run()`.  
This is the recommended path for creating a new collection from scratch — the Ingestor handles collection creation, chunking, embedding lookup, and indexing in one shot.

In [ ]:
collection_name = "json_multi_file_collection"
existing = Collection(name=collection_name)
if existing.exists:
    existing.destroy()

In [7]:
ingestor = (
    Ingestor(
        name=collection_name,
        type=CollectionType.FILE_EMBEDDING_BASED,
        description="Multi-file JSON ingestion from local directory"
    )
    .files(
        files=local_config,
        ingestor=BasicIngestor(chunk_size=512, chunk_overlap=50),
        extraction_schema=extraction_schema
    )
    .upsert(embedding_model=embedding_model)
    .run()
)

Initializing collection pipeline...


/usr/local/lib/python3.12/dist-packages/teradatagenai/vector_store/collection.py:649: UserWarning: Collection does not exist or name is not supplied. Create it before proceeding ahead.
  warnings.warn("Collection does not exist or name is not supplied. Create it before proceeding ahead.")


Creating collection...
Collection initialized successfully
Starting create collection operation...
Create Collection completed.                                                                          
Completed: ｜⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿｜ 100% - 100/100              
Create Collection completed successfully
Processing files...
Files uploaded successfully and ingestion in progress
Starting ingest operation...
Ingest completed.                                                                          
Completed: ｜⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿｜ 100% - 100/100   
Ingest completed successfully
Collection update request is accepted and in progress
Starting update operation...
Update completed.                                                                          
Completed: ｜⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿｜ 100% - 720/720   
Update completed successfully
Pipeline completed successfully! Operati

#### Inspect per-file ingestion status

After `.run()`, call `ingestor.get_file_metadata()` to see the ingestion status of every file.  
If any file shows a status other than `SUCCESS`, the metadata will tell you what went wrong (parse error, missing field, upload failure, etc.).

In [9]:
ingestor_check = Ingestor(name=collection_name)
files_metadata = ingestor_check.get_file_metadata(return_type="json")
# returns a JSON string — display raw or parse below
files_metadata

Collection json_multi_file_collection initialized for use.


{'file_metadata_count': 25,
 'page': 1,
 'page_size': 25,
 'file_metadata_list': [{'collection_name': 'json_multi_file_collection',
   'object_name': '"vsdemo03"."ingest_table_ad688785f2e04d159032d5d4c1bd17c7"',
   'file_name': 'alg-geom9302003.json',
   'md5_hash': '76bfd0e4b70915a7b66631b68422e6f3',
   'status': 'ready',
   'error_message': None,
   'last_updated': '2026-06-02 11:17:59.829496+00:00'},
  {'collection_name': 'json_multi_file_collection',
   'object_name': '"vsdemo03"."ingest_table_ad688785f2e04d159032d5d4c1bd17c7"',
   'file_name': 'alg-geom9309007.json',
   'md5_hash': 'ac4fb858acea0040da4dc152206c0b1b',
   'status': 'ready',
   'error_message': None,
   'last_updated': '2026-06-02 11:17:59.829496+00:00'},
  {'collection_name': 'json_multi_file_collection',
   'object_name': '"vsdemo03"."ingest_table_ad688785f2e04d159032d5d4c1bd17c7"',
   'file_name': 'alg-geom9601014.json',
   'md5_hash': 'cfd3fc011a2fd0ed0b72c1d72d43e184',
   'status': 'ready',
   'error_message': N

In [10]:
# get_file_metadata returns a dict with a 'file_metadata_list' key
parsed = json.loads(files_metadata) if isinstance(files_metadata, str) else files_metadata
metadata_list = parsed.get("file_metadata_list", parsed) if isinstance(parsed, dict) else parsed

failed = [f for f in metadata_list if f.get("status") != "ready"]
if failed:
    print(f"{len(failed)} file(s) did not reach 'ready' status:")
    for f in failed:
        print(f"  {f.get('file_name')} — status: {f.get('status')} | {f.get('error_message', 'no detail')}")
else:
    print(f"All {len(metadata_list)} files ingested successfully.")

1 file(s) did not reach 'ready' status:
  adap-org9810002.json — status: ingestion_failed | Available keys in file: ['data_source', 'date_modified', 'date_processed', 'element_id', 'embeddings', 'filename', 'filetype', 'languages22', 'metadata', 'mode', 'orig_elements', 'page_number', 'path', 'permissions_data', 'record_locator', 'text', 'type'], Missing: ['languages']


To retrieve information about storage objects (tables) containing ingested file data use the get_file_store method

In [13]:
files_store = ingestor_check.get_file_store(return_type="json")
files_store

{'file_store_count': 1,
 'page': 1,
 'page_size': 1,
 'file_store_list': [{'collection_name': 'json_multi_file_collection',
   'object_name': '"vsdemo03"."ingest_table_ad688785f2e04d159032d5d4c1bd17c7"',
   'file_params': {'files_parameters': {'files_type': 'json',
     'delimiter': None,
     'json_source_type': 'default'},
    'storage_location': None,
    'ingest_parameters': None,
    'extraction_schema': {'table_name': 'ingest_table_ad688785f2e04d159032d5d4c1bd17c7',
     'key_columns': [{'name': 'TD_FILENAME',
       'datatype': 'VARCHAR(1024)',
       'key_name': None}],
     'metadata_columns': [{'name': 'languages',
       'datatype': None,
       'key_name': None}],
     'data_columns': [{'name': 'text',
       'datatype': 'varchar(32000)',
       'key_name': None}],
     'image_column': None,
     'embedding_columns': [{'name': 'embeddings',
       'datatype': None,
       'key_name': None}]},
    'overwrite_files': False,
    'overwrite_object': False},
   'status': 'ready'

---
## <b style='font-size:22px;font-family:Arial'>Approach 2 — Empty Collection + `add_documents`</b>

Use this pattern when you want to initialize an empty collection first and then populate it separately — for example, to add files incrementally or from multiple sources in different calls.

Steps:
1. Create an empty `Ingestor()` (no `.files()`) and call `.run()` to register the collection
2. Wrap it as a `Collection` object
3. Call `coll.add_documents(documents=local_config, ...)` to append files

In [5]:
alt_collection_name = "json_multi_file_collection_alt"

# Step 1: Create an empty collection (no files attached)
empty_ingestor = Ingestor(
    name=alt_collection_name,
    type=CollectionType.FILE_EMBEDDING_BASED,
    description="Alternate collection — populated via add_documents"
)
empty_ingestor.run()

Initializing collection pipeline...


/usr/local/lib/python3.12/dist-packages/teradatagenai/vector_store/collection.py:649: UserWarning: Collection does not exist or name is not supplied. Create it before proceeding ahead.
  warnings.warn("Collection does not exist or name is not supplied. Create it before proceeding ahead.")


Creating collection...
Collection initialized successfully
Starting create collection operation...
Create Collection completed.                                                                          
Completed: ｜⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿｜ 100% - 100/100              
Create Collection completed successfully
Pipeline completed successfully! Operations: create_collection


{'status': {'success': True,
  'errors': [],
  'warnings': [],
  'message': 'Pipeline executed successfully'},
 'collection': 'json_multi_file_collection_alt',
 'operation_details': {'create_collection': {'success': True,
   'api_response': 'Collection initialized successfully',
   'status_result': {'success': True,
    'status_response': {'collection_name': 'json_multi_file_collection_alt',
     'collection_status': 'initialized'}}}}}

In [6]:
# Step 2: Reference it as a Collection object
coll_alt = Collection(name=alt_collection_name)

# Step 3: Add files — LocalConfig is passed directly, same as in the Ingestor pipeline
coll_alt.add_documents(
    documents=local_config,
    extraction_schema=extraction_schema,
    #embedding=embedding_model --> Not required for FILE_EMBEDDING_BASED 
)

Collection json_multi_file_collection_alt initialized for use.
Initializing collection pipeline...
Collection json_multi_file_collection_alt initialized for use.
Creating collection...
Processing files...
Files uploaded successfully and ingestion in progress
Starting ingest operation...
Ingest completed.                                                                          
Completed: ｜⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿｜ 100% - 100/100   
Ingest completed successfully
Collection update request is accepted and in progress
Starting update operation...
Update completed.                                                                          
Completed: ｜⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿｜ 100% - 720/720   
Update completed successfully
Pipeline completed successfully! Operations: create_collection, ingest, create_index


---
## <b style='font-size:22px;font-family:Arial'>5 · Explore Ingested Data</b>

`get_indexes_embeddings()` returns the full index table — useful for verifying what was ingested, checking field values, and spotting anomalies.


In [13]:
coll = Collection(name=alt_collection_name)
df = coll.get_indexes_embeddings()
df.head(1)

Collection json_multi_file_collection_alt initialized for use.


DataBaseName,TableName,TD_ID,td_filename,languages,text,vector_index,vector_index_normalized
td_genai_user,ingest_table_fd5465f8ce0c4ae29139df76923c9a92,265,alg-geom9412017.json,"[""eng""]","Consider the Koszul resolution of OD(v)∩ tral sequence degenerates in ′′E2, because ′′Ep,q for p 6= r, and ′′E0,r 1 V . Then the corresponding second spec- 1 = 0 for q 6= d − s − 1,0, ′′E0,p 1 = 0 b ∼= C. Now the statements (i)-(iii) are obvious. ✷ 21 Corollary 8.4 Let assumptions in 8.3, one obtains V be a complete intersection such that d − r ≥ 3. Then, under b hd−r−1(Ω1 V ) = b r X i=1 X J⊂I (−1)r−|J|l∗(∆i + X j∈J ∆j) − d− − X dimΘ=0 Θ⊂∆∗ X J⊂I (−1)r−|J|l∗(X","0.005218505859375,0.0137405395507812,0.040313720703125,0.033050537109375,0.0182647705078125,0.033660888671875,0.009307861328125,0.0057525634765625,-0.008880615234375,0.0277557373046875,0.028839111328125,-0.02606201171875,-0.024322509765625,0.00451278686523438,0.046417236328125,0.0155868530273438,0.0280914306640625,0.0115509033203125,0.0250396728515625,0.082763671875,0.00788116455078125,-0.04144287109375,0.0170745849609375,-0.01885986328125,0.0060882568359375,-0.034942626953125,0.039794921875,0.052520751953125,0.0699462890625,-0.045318603515625,0.00942230224609375,-0.046295166015625,0.018646240234375,-0.0457763671875,0.003753662109375,0.059600830078125,-0.0084991455078125,0.03857421875,-0.0144882202148438,0.0291290283203125,0.040008544921875,0.0335693359375,-0.0504150390625,-0.0114822387695312,0.01885986328125,0.02410888671875,-0.005767822265625,0.0221099853515625,-0.0145263671875,-0.027679443359375,0.00415802001953125,0.049774169921875,0.0191497802734375,0.0095672607421875,-0.06536865234375,-0.031143188476562","0.00521676428616047,0.0137359546497464,0.0403002686798573,0.0330395065248013,0.0182586759328842,0.0336496569216251,0.00930475536733866,0.00575064402073622,-0.00887765176594257,0.0277464743703604,0.0288294870406389,-0.0260533150285482,-0.0243143923580647,0.00451128091663122,0.046401746571064,0.0155816515907645,0.0280820559710264,0.0115470485761762,0.025031317025423,0.0827360525727272,0.00787853449583054,-0.0414290428161621,0.017068887129426,-0.0188535694032907,0.00608622515574098,-0.0349309667944908,0.039781641215086,0.0525032244622707,0.0699229463934898,-0.0453034788370132,0.00941915810108185,-0.0462797172367573,0.0186400171369314,-0.045761089771986,0.00375240948051214,0.0595809407532215,-0.0084963096305728,0.0385613478720188,-0.0144833857193589,0.0291193071752787,0.0399951934814453,0.0335581339895725,-0.0503982156515121,-0.0114784073084593,0.0188535694032907,0.0241008419543505,-0.00576589768752456,0.0221026074141264,-0.0145215196534991,-0.02767020650208,0.00415663234889507,0.0497575588524342,0.01914338953793"


In [ ]:
# List unique source files that were ingested and check if the corrupted file is among them
pdf = df.to_pandas()
ingested_files = pdf["td_filename"].unique()
print("Ingested files:")

target = "adap-org9810002.json"
if target in ingested_files:
    print(f"\n✓ '{target}' is present in the ingested files.")
else:
    print(f"\n✗ '{target}' was NOT found in the ingested files.")

Ingested files:

✗ 'adap-org9810002.json' was NOT found in the ingested files.


In [ ]:
fixed_file_path = os.path.join(os.path.dirname(notebook_dir), "example-data", "json_files", "adap-org9810002_fixed.json")
if os.path.exists(fixed_file_path):
    print(f"✓ Fixed file found at: {fixed_file_path}")

In [ ]:
local_config_fixed_file = LocalConfig(files = [fixed_file_path],
                                      files_type="json")

In [ ]:
#Fix the file and add it to the existing collection
coll_alt.add_documents(
    documents=local_config_fixed_file,
    extraction_schema=extraction_schema,
)

Initializing collection pipeline...
Collection json_multi_file_collection_alt initialized for use.
Creating collection...
Processing files...
Files uploaded successfully and ingestion in progress
Starting ingest operation...
Ingest completed.                                                                          
Completed: ｜⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿｜ 100% - 100/100   
Ingest completed successfully
Collection update request is accepted and in progress
Starting update operation...
Update completed.                                                                          
Completed: ｜⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿｜ 100% - 720/720   
Update completed successfully
Pipeline completed successfully! Operations: create_collection, ingest, create_index


In [ ]:
# Check if the file is present in the collection
ing = Ingestor(name=alt_collection_name)

ing.get_file_metadata(return_type="json")

Collection json_multi_file_collection_alt initialized for use.


{'file_metadata_count': 26,
 'page': 1,
 'page_size': 26,
 'file_metadata_list': [{'collection_name': 'json_multi_file_collection_alt',
   'object_name': '"td_genai_user"."ingest_table_330208b2f4114ab9a034717b705512fc"',
   'file_name': 'adap-org9810002_fixed.json',
   'md5_hash': '60c123ec3c8343005872742fc39eb860',
   'status': 'ready',
   'error_message': None,
   'last_updated': '2026-05-29 09:02:15.415453+00:00'},
  {'collection_name': 'json_multi_file_collection_alt',
   'object_name': '"td_genai_user"."ingest_table_fd5465f8ce0c4ae29139df76923c9a92"',
   'file_name': 'adap-org9812002.json',
   'md5_hash': '1bac03774e8ba3ef1efe5c3095481f40',
   'status': 'ready',
   'error_message': None,
   'last_updated': '2026-05-29 08:32:16.709619+00:00'},
  {'collection_name': 'json_multi_file_collection_alt',
   'object_name': '"td_genai_user"."ingest_table_fd5465f8ce0c4ae29139df76923c9a92"',
   'file_name': 'alg-geom9601006.json',
   'md5_hash': 'be62cd8f0a7473fca4f30efd6bc9948b',
   'status

---
## <b style='font-size:22px;font-family:Arial'>6 · Delete Records</b>

Two deletion methods are available:

| Method | What it removes |
|---|---|
| `delete_documents(file_name)` | Delete the entire file and all records from that file |
| `delete_by_ids(ids=[...])` | Specific rows by their `TD_ID` primary key |

In [15]:
# Delete all records from a specific source file
coll.delete_documents(documents = ["adap-org9810002.json"])

Collection update request is accepted and in progress


In [16]:
coll.status()

collection_name,collection_status,retry_after
json_multi_file_collection_alt,update_finishing,60


In [18]:
target_file = "adap-org9810002.json"

df_after = coll.get_indexes_embeddings()
remaining = df_after.to_pandas()
still_present = remaining[remaining["td_filename"] == target_file]

if still_present.empty:
    print(f"'{target_file}' has been successfully removed from the collection.")
else:
    print(f"'{target_file}' still has {len(still_present)} record(s) in the collection.")

'adap-org9810002.json' has been successfully removed from the collection.


In [19]:
# Delete specific records by TD_ID
# Replace with actual IDs found via get_indexes_embeddings()
coll.delete_by_ids(ids=[1407])

Collection update request is accepted and in progress


---
## <b style='font-size:22px;font-family:Arial'>7 · Similarity Search</b>

Returns the top-K most semantically similar records to a natural language question.

In [20]:
coll.similarity_search(
    question="What are the main topics discussed in these papers?",
    top_k=3,
    embedding_model=embedding_model,
)

similar_objects_count:3
similar_objects:
      score DataBaseName                                      TableName  TD_ID           td_filename                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                          text  index_label
0  0.069604     vsdemo04  ingest_table_2a89b41d9aa24a1a9b7f12c633a7b8b2   2802  alg-geom9607012.json                                                                                                                                                                                                                                                           

---
## <b style='font-size:22px;font-family:Arial'>8 · Ask a Question (RAG)</b>

`ask()` combines similarity search with an LLM to generate a grounded natural language answer from your collection.  
Configure a `chat_model` (TeradataAI) and the collection will retrieve relevant context, then pass it to the LLM.

In [ ]:
# Ask requires a chat-capable model, not just an embedding model. 
# To view a list of hosted embedidng and chat models users can use list_available_models method in  CollectionManager
# Alternaltively, users can specify external models as well.
CollectionManager.list_available_models()

embedding_models:
                             model_id                      model_name provider  status
0     amazon.titan-embed-text-v1:2:8k      Titan Embeddings G1 - Text   Amazon  ACTIVE
1        amazon.titan-embed-text-v2:0        Titan Text Embeddings V2   Amazon  ACTIVE
2       amazon.titan-embed-image-v1:0  Titan Multimodal Embeddings G1   Amazon  ACTIVE
3         amazon.titan-embed-image-v1  Titan Multimodal Embeddings G1   Amazon  ACTIVE
4             cohere.embed-english-v3                   Embed English   Cohere  ACTIVE
5  cohere.embed-multilingual-v3:0:512              Embed Multilingual   Cohere  ACTIVE
6       cohere.embed-english-v3:0:512                   Embed English   Cohere  ACTIVE
7          amazon.titan-embed-text-v1      Titan Embeddings G1 - Text   Amazon  ACTIVE
8       amazon.titan-embed-g1-text-02        Titan Text Embeddings v2   Amazon  ACTIVE
9                   cohere.embed-v4:0                        Embed v4   Cohere  ACTIVE

chat_models:
           

In [24]:
chat_model = TeradataAI(
    api_type="td_hosted",
    model_name="openai.gpt-oss-120b-1:0",
)

answer = coll.ask(
    question="What are the main topics discussed in these papers?",
    embedding_model=embedding_model,
    chat_model=chat_model,
    top_k=5
)
print(answer)

The excerpts come from a collection of algebra‑geometry papers that cover several closely‑related research themes:

* **Operator theory in tensor algebras** – One paper proves a characterization of when two non‑degenerate operators \(A,B\in A_n\otimes\mathbb Q\) lie in the same orbit under the natural action of the isometry group, and it introduces a rank functional \( \operatorname{rk}:A_n\to\mathbb Z\).

* **Group‑theoretic classifications** – Another work revisits the case of a group \(G\) of type \((3,1)\) whose stabiliser in \(\mathrm{SL}(3)\) is irreducible and contains a \(\mathbb Z_3\) subgroup, a situation that is linked to a broader conjecture (Conjecture 1).

* **Hermite‑type results** – One short entry simply mentions “Hermite,” indicating a discussion of Hermite normal form or related lattice‑theoretic concepts.

* **Singularities of plane curves** – Two papers focus on curve singularities:
  * The semigroup of values of a curve singularity with several branches (Delgado &

---
## <b style='font-size:22px;font-family:Arial'>9 · Destroy Collections</b>

Clean up both collections from Teradata Vantage when they are no longer needed.

In [ ]:
coll.destroy()
coll_alt.destroy()